# ITCC508 Lab Exercise PT-F2 — Optimizing Chunk Size and Overlap (Member 3 dataset)

**Student:** Audrey Benjamin Gutierrez · Section 401 I · Group QuantumStack, Member 3
**Course:** ITC-C508 — Deep Learning (NLP)

---

## Dataset

**Source document.** `LendSuri_Member_3_Corpus_clean.pdf` — the cleaned Member 3 corpus of the LendSuri PT-F1 single-PDF package. It holds **126 pages, one researcher-prepared record per page**, summarising Philippine SEC records on lending companies and online lending platforms (OLPs):

| Pages | Record type | Date |
| --- | --- | --- |
| 1–60 | Current registry snapshot of recorded OLPs | 2026-01-26 |
| 61–101 | Historical list of recorded OLPs | 2021-11-29 |
| 102–123 | SEC regulatory events and advisories (revocations, impersonation, unauthorized use of names) | 2023–2026 |
| 124–125 | SEC framework and guidance | 2025–2026 |
| 126 | SEC complaint guidance | 2026-10-05 |

**Text layer.** Every page has a text layer: `pypdf` extracts all 126 pages, so no OCR was needed.

**What was cleaned.** Every page of the package file carries the same four running elements: the line `LendSuri | Researcher-prepared record from the existing v5 corpus`, a printed page number that belongs to the *original category PDF* rather than to this file (the package README warns not to use it), the label `LendSuri / Member 3`, and a two-line disclaimer footer. Together they are **34,368 of 164,290 characters — 20.9% of the corpus**, identical on all 126 pages. Left in, they would bias the experiment: at 300-character chunks the boilerplate fills a large share of each chunk, while at 1800 it is a small fraction of a whole page, so pollution falls as chunk size grows for reasons unrelated to retrieval. The printed page number would also put a misleading page reference into the indexed text.

These elements were removed and the remaining lines re-rendered, one record per page, as **`LendSuri_Member_3_Corpus_clean.pdf`** (126 pages, 129,983 characters after this pipeline's whitespace normalisation). Each record stays on the same page number as in the package file, so the package's gold page numbers apply unchanged. `LendSuri_Member_3_Corpus.pdf` is kept unmodified as the raw control.

The answer keys, question sheet and page map that come with the package are **never** uploaded or indexed — they are used only to define the ground truth below.

**Dataset characteristic that matters for chunking.** Even after cleaning, each page repeats the same record template — header fields, a "RAG source text" body and an "Interpretation limits" note — so most of every page is wording shared by all records. Only the platform name, company name, dates and advisory title distinguish one record from another.

**Known limitation — report, do not fix.** `all-MiniLM-L6-v2` truncates input at 256 word-pieces. At `CHUNK_SIZE = 1800` (a whole page per chunk), 69 of the 126 cleaned pages exceed 256 word-pieces, so the end of those pages is not embedded. Every evidence passage used for scoring lies inside the embedded part.

---

## Test questions and ground truth

Nine questions, three per difficulty level. M3-E, M3-A and M3-D are the package's Member 3 questions, with the answer key's gold pages; the other six were written from the Member 3 corpus and verified before any configuration was run.

| ID | Difficulty | Question (short) | Gold page(s) |
| --- | --- | --- | --- |
| M3-E | Easy | Company Peso2you is listed under | 46 |
| E2 | Easy | Company Atome PH is recorded under in January 2026 | 35 |
| E3 | Easy | SEC department for complaints about lending without a Certificate of Authority | 126 |
| M3-A | Average | BillEase / First Digital Finance and the SEC warning about its name | 15, 109 |
| A2 | Average | Mabilis Cash: company and the dates of both lists | 82, 32 |
| A3 | Average | The June 2025 landline order, and whether it proves a platform's status | 124 (two passages) |
| M3-D | Difficult | Tala's listing and the two warnings about fake pages and apps | 54, 105, 111 |
| D2 | Difficult | ACOM: operator, both lists, and the fake-Facebook-page advisory | 61, 1, 119 |
| D3 | Difficult | Snapcash's 2021 platforms vs. Peso2you and Jetpeso in January 2026 | 95, 46, 49 |

Eighteen evidence units in total. Each is the exact answer-bearing passage on its gold page, stored as text and located at run time in the pipeline's own whitespace-normalised page text; the notebook stops if any passage is not found exactly once.

---

## Notebook structure

Per Step 1 of Part 1, each original code cell is split into logical functional sections with a Markdown explanation above it. Component I (vector indexing) is the provided pipeline with the Member 3 corpus. Component II (prompt and generation) keeps the Part 1 changes for the assigned model, with the prompt instruction rewritten for this dataset. Component III is the Part 2 experiment: twelve chunking configurations, nine questions, Precision@3, Recall@3 and MRR@3, the results workbook and the detailed retrieval log.

# Component I. Vector Indexing and Embedding

### 1. Install required libraries

Installs the five packages the indexing pipeline depends on, with `-q` to suppress pip's output. `faiss-cpu` provides the vector index; `sentence-transformers` provides the embedding model; `pypdf` reads the PDF; `transformers` and `sentencepiece` are required by the language model in Component II; `pandas` and `openpyxl` write the chunk-inspection workbook.

Colab discards installed packages when the runtime is recycled, so this cell must run first in every new session.

In [1]:
# ============================================================
# RAG DOCUMENT PROCESSING PIPELINE
# PDF EXTRACTION, CHUNKING, EMBEDDINGS, FAISS, AND RETRIEVAL
# WITH DOWNLOADABLE EXCEL CHUNK INSPECTION
# Google Colab implementation
# ============================================================


# ============================================================
# 1. INSTALL REQUIRED LIBRARIES
# ============================================================

!pip install -q \
    faiss-cpu \
    sentence-transformers \
    pypdf \
    transformers \
    sentencepiece \
    pandas \
    openpyxl

### 2. Import libraries

Brings the installed packages into scope. `files` is Colab's upload/download bridge. `PdfReader` performs text extraction, `SentenceTransformer` produces embeddings, and `faiss` holds the vector index.

`re` is used for whitespace normalisation during extraction, `pickle` to persist the chunk list and configuration, and the `openpyxl.styles` imports to format the Excel inspection sheet.

In [2]:
# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

from google.colab import files
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

import faiss
import numpy as np
import pandas as pd
import pickle
import re
import os

from openpyxl.styles import Font, PatternFill, Alignment

### 3. Indexing parameters

**This is the only cell to edit between manual runs.**

`CHUNK_SIZE` and `CHUNK_OVERLAP` are measured in characters, not tokens. They are set to the Part 2 baseline, 1000 / 200. `TOP_K` fixes retrieval depth at 3, which is what Precision@3, Recall@3 and MRR@3 are defined against — leave it alone.

`EMBEDDING_MODEL_NAME` selects `all-MiniLM-L6-v2`, which produces 384-dimensional vectors. This model truncates its input at 256 word-pieces, so at `CHUNK_SIZE = 1800` the end of 69 of the 126 pages is never embedded. Record that effect rather than working around it.

Component III does not read these values: it builds its own chunks and its own index for each of the twelve configurations.

In [3]:
# ============================================================
# 3. INDEXING PARAMETERS
# ============================================================

# Change these values to experiment with chunking.

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 3

# Embedding model
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

### 4. Validate chunk parameters

Guards against configurations that would silently produce nonsense before any expensive work happens.

The important check is `CHUNK_OVERLAP >= CHUNK_SIZE`: the chunking loop advances by `step_size = chunk_size - chunk_overlap`, so an overlap equal to or larger than the chunk size yields a step of zero or less and an infinite loop. Failing loudly here is cheaper than hanging the runtime.

The cell then prints the active configuration, which is worth capturing in your results table for each run.

In [4]:
# ============================================================
# 4. VALIDATE CHUNK PARAMETERS
# ============================================================

if CHUNK_SIZE <= 0:
    raise ValueError("CHUNK_SIZE must be greater than 0.")

if CHUNK_OVERLAP < 0:
    raise ValueError("CHUNK_OVERLAP cannot be negative.")

if CHUNK_OVERLAP >= CHUNK_SIZE:
    raise ValueError(
        "CHUNK_OVERLAP must be smaller than CHUNK_SIZE."
    )

if TOP_K <= 0:
    raise ValueError("TOP_K must be greater than 0.")


print("=" * 70)
print("RAG INDEXING CONFIGURATION")
print("=" * 70)

print("Chunk size       :", CHUNK_SIZE)
print("Chunk overlap    :", CHUNK_OVERLAP)
print("Top-K             :", TOP_K)
print("Embedding model   :", EMBEDDING_MODEL_NAME)

RAG INDEXING CONFIGURATION
Chunk size       : 1000
Chunk overlap    : 200
Top-K             : 3
Embedding model   : all-MiniLM-L6-v2


### 5. Upload the PDF document

Opens Colab's file picker and blocks until a file is chosen. Only the first uploaded file is used, and a non-`.pdf` extension raises immediately.

**Upload `LendSuri_Member_3_Corpus_clean.pdf` here** — and only that file. The package's answer key, question sheet, gold-evidence list and page map must not be uploaded: indexing them would leak the answers into the vector store.

This cell is interactive, so it must be re-run manually each time the runtime restarts.

In [5]:
# ============================================================
# 5. UPLOAD PDF DOCUMENT
# ============================================================

print("\n")
print("=" * 70)
print("UPLOAD DOCUMENT")
print("=" * 70)

uploaded = files.upload()

if len(uploaded) == 0:
    raise RuntimeError("No file was uploaded.")

# Use the first uploaded PDF
pdf_file = next(iter(uploaded.keys()))

if not pdf_file.lower().endswith(".pdf"):
    raise ValueError(
        "The uploaded file is not a PDF. Please upload a PDF."
    )

print("\nSelected file:")
print(pdf_file)



UPLOAD DOCUMENT


Saving LendSuri_Member_3_Corpus_clean.pdf to LendSuri_Member_3_Corpus_clean.pdf

Selected file:
LendSuri_Member_3_Corpus_clean.pdf


### 6. Extract text from the PDF, and verify something came out

Iterates the PDF's pages and calls `extract_text()` on each, keeping a `{page, text}` record for every page that yields content. Pages extracting to nothing are skipped, so `pages` may be shorter than the document.

`re.sub(r"\s+", " ", page_text)` collapses every run of whitespace — including newlines — into a single space. **This is why evidence passages are located per page:** the pipeline never sees the document as one continuous string, and Component III searches for each evidence passage inside this normalised page text.

The guard that follows raises if no page produced text. The cleaned Member 3 corpus has a text layer on all 126 pages, so it passes; a scanned PDF would stop here and need OCR.

In [6]:
# ============================================================
# 6. EXTRACT TEXT FROM PDF
# ============================================================

print("\n")
print("=" * 70)
print("EXTRACTING PDF TEXT")
print("=" * 70)

reader = PdfReader(pdf_file)

pages = []

for page_number, page in enumerate(reader.pages, start=1):

    page_text = page.extract_text()

    if page_text:

        # Normalize whitespace
        page_text = re.sub(
            r"\s+",
            " ",
            page_text
        ).strip()

        if page_text:
            pages.append({
                "page": page_number,
                "text": page_text
            })


print("PDF pages:", len(reader.pages))
print("Pages containing text:", len(pages))


# ============================================================
# 7. CHECK WHETHER TEXT WAS EXTRACTED
# ============================================================

if len(pages) == 0:
    raise RuntimeError(
        "No text could be extracted from the PDF. "
        "The PDF may consist of scanned images and require OCR."
    )



EXTRACTING PDF TEXT
PDF pages: 126
Pages containing text: 126


### 7. Create document chunks

Splits each page into fixed-size character windows, advancing by `step_size = chunk_size - chunk_overlap` so consecutive chunks overlap by the configured amount.

**Chunking runs independently per page.** The loop restarts at `start = 0` for every page, so no chunk ever spans a page boundary, and each page contributes a short final chunk from its remainder. This inflates chunk counts relative to chunking the document as a whole, and it means evidence split across a page break can never appear in a single chunk.

Each chunk carries its `chunk_id`, source `page`, and text. Empty chunks are discarded rather than indexed.

In [7]:
# ============================================================
# 8. CREATE DOCUMENT CHUNKS
# ============================================================

def create_chunks(
    pages,
    chunk_size,
    chunk_overlap
):
    """
    Split each page into character-based chunks.

    Chunking is performed independently for each page.
    Overlap is maintained between consecutive chunks
    from the same page.
    """

    chunks = []
    chunk_id = 0

    step_size = chunk_size - chunk_overlap

    for page_data in pages:

        page_number = page_data["page"]
        page_text = page_data["text"]

        start = 0

        while start < len(page_text):

            end = start + chunk_size

            chunk_text = page_text[start:end].strip()

            if chunk_text:
                chunks.append({
                    "chunk_id": chunk_id,
                    "page": page_number,
                    "text": chunk_text
                })

                chunk_id += 1

            # Move forward while maintaining overlap
            start += step_size

    return chunks


chunks = create_chunks(
    pages,
    CHUNK_SIZE,
    CHUNK_OVERLAP
)

### 8. Display chunking information

Reports the chunk count and the mean, minimum and maximum chunk lengths, then raises if no chunks were produced.

The minimum is the diagnostic worth watching: because chunking is per page, the shortest chunk is a page remainder and may be only a few characters. A large gap between the mean and the configured `CHUNK_SIZE` indicates many such partial chunks, which is expected at small chunk sizes.

Record the chunk count for every run — it is part of the efficiency evidence the exercise asks for.

In [8]:
# ============================================================
# 9. DISPLAY CHUNKING INFORMATION
# ============================================================

print("\n")
print("=" * 70)
print("CHUNKING RESULTS")
print("=" * 70)

print("Number of chunks:", len(chunks))

if len(chunks) == 0:
    raise RuntimeError(
        "No chunks were generated. Check the extracted PDF text."
    )

chunk_lengths = [
    len(chunk["text"])
    for chunk in chunks
]

print(
    "Average characters per chunk:",
    round(np.mean(chunk_lengths), 2)
)

print(
    "Shortest chunk:",
    min(chunk_lengths),
    "characters"
)

print(
    "Longest chunk:",
    max(chunk_lengths),
    "characters"
)



CHUNKING RESULTS
Number of chunks: 235
Average characters per chunk: 627.57
Shortest chunk: 2 characters
Longest chunk: 1000 characters


### 9. Export the generated chunks to Excel

Writes one row per chunk to a formatted workbook and triggers a browser download.

Each row carries the chunk ID, source document, page number, character count, the `CHUNK_SIZE` and `CHUNK_OVERLAP` that produced it, and the chunk text. Embedding the settings in every row means workbooks from different runs stay identifiable after download.

The formatting block styles the header, sets column widths, enables autofilter and freezes the header row. **Use this workbook to mark ground truth:** filter to the chunks containing each test question's answer and record their chunk IDs, as Lecture 8 slide 13 describes.

In [9]:
# ============================================================
# 10. EXPORT GENERATED CHUNKS TO EXCEL
# ============================================================

print("\n")
print("=" * 70)
print("EXPORTING GENERATED CHUNKS TO EXCEL")
print("=" * 70)

# Prepare one Excel row for every generated chunk.
chunk_records = []

for chunk in chunks:

    chunk_records.append({
        "Chunk_ID": chunk["chunk_id"],
        "Source_Document": pdf_file,
        "Page_Number": chunk["page"],
        "Character_Count": len(chunk["text"]),
        "Chunk_Size_Setting": CHUNK_SIZE,
        "Chunk_Overlap_Setting": CHUNK_OVERLAP,
        "Chunk_Text": chunk["text"]
    })

chunks_df = pd.DataFrame(chunk_records)

# Create an output filename based on the source PDF.
base_name = os.path.splitext(
    os.path.basename(pdf_file)
)[0]

EXCEL_FILE = f"{base_name}_chunks_inspection.xlsx"

# Create and format the Excel workbook.
with pd.ExcelWriter(
    EXCEL_FILE,
    engine="openpyxl"
) as writer:

    chunks_df.to_excel(
        writer,
        sheet_name="Generated_Chunks",
        index=False
    )

    worksheet = writer.sheets["Generated_Chunks"]

    # Format the header row.
    for cell in worksheet[1]:

        cell.font = Font(
            bold=True,
            color="FFFFFF"
        )

        cell.fill = PatternFill(
            fill_type="solid",
            fgColor="1F4E78"
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

    # Set column widths.
    column_widths = {
        "A": 12,
        "B": 35,
        "C": 14,
        "D": 18,
        "E": 20,
        "F": 23,
        "G": 80
    }

    for column, width in column_widths.items():
        worksheet.column_dimensions[column].width = width

    # Format rows for easier inspection.
    for row in worksheet.iter_rows(
        min_row=2,
        max_row=worksheet.max_row
    ):

        for cell in row:

            cell.alignment = Alignment(
                vertical="top",
                wrap_text=(cell.column == 7)
            )

    # Keep headers visible while scrolling.
    worksheet.freeze_panes = "A2"

    # Enable filtering.
    worksheet.auto_filter.ref = worksheet.dimensions

    # Set header height.
    worksheet.row_dimensions[1].height = 32

print("Excel file created:", EXCEL_FILE)
print("Chunks exported:", len(chunks_df))
print("Excel worksheet: Generated_Chunks")

print("\nExcel export preview:")

print(
    chunks_df[
        [
            "Chunk_ID",
            "Page_Number",
            "Character_Count"
        ]
    ].head(10).to_string(index=False)
)

# Automatically download the Excel workbook in Google Colab.
files.download(EXCEL_FILE)



EXPORTING GENERATED CHUNKS TO EXCEL
Excel file created: LendSuri_Member_3_Corpus_clean_chunks_inspection.xlsx
Chunks exported: 235
Excel worksheet: Generated_Chunks

Excel export preview:
 Chunk_ID  Page_Number  Character_Count
        0            1             1000
        1            1              378
        2            2             1000
        3            2              448
        4            3             1000
        5            3              611
        6            4             1000
        7            4              450
        8            5             1000
        9            5              313


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 10. Display sample chunks

Prints the first three chunks in full as a sanity check on extraction and splitting.

Read these rather than skimming them. Garbled text, missing spaces between words, or chunks that begin mid-sentence all surface here, before embedding cost is incurred. Chunks starting mid-word are normal for character-based splitting and are not a fault.

In [10]:
# ============================================================
# 11. DISPLAY SAMPLE CHUNKS
# ============================================================

print("\n")
print("=" * 70)
print("SAMPLE CHUNKS")
print("=" * 70)

for chunk in chunks[:3]:

    print(f"\nChunk ID : {chunk['chunk_id']}")
    print(f"Page     : {chunk['page']}")
    print(f"Characters: {len(chunk['text'])}")

    print("-" * 70)

    print(chunk["text"])



SAMPLE CHUNKS

Chunk ID : 0
Page     : 1
Characters: 1000
----------------------------------------------------------------------
A Current OLP registry snapshot (2026-01-26) Record ID: CUR2026_0001 Record date: 2026-01-26 Platform or entity: ACOM – Fast Cash Loan. Simple. Company: Acom Consumer Finance Corporation Evidence category: CURRENT_RECORDED_OLP_SNAPSHOT_2026 Classification: RECORDED_IN_2026_01_26_SNAPSHOT Evidence priority: CURRENT_POSITIVE_SNAPSHOT RAG source text (v5 normalized text) SEC OLP current-registry snapshot dated 2026-01-26. Platform: ACOM – Fast Cash Loan. Simple.. Company: Acom Consumer Finance Corporation. SEC registration no.: CS201724897. Form 1 submission date(s): 01 October, 2019. Remarks: none stated. IMPORTANT: use this as dated 2026 positive registry evidence. Later SEC enforcement/advisory evidence overrides older positive evidence when relevant. Absence from this snapshot must not be treated as proof of unauthorized status. Interpretation limits: Reco

### 11. Load the embedding model

Downloads `all-MiniLM-L6-v2` from Hugging Face on first use and caches it for the session. Expect a short delay on the first run.

This model maps text to 384-dimensional vectors and has a maximum input of 256 word-pieces. Anything longer is truncated silently — no warning is raised — so larger chunk settings are embedded only in part.

In [11]:
# ============================================================
# 12. LOAD EMBEDDING MODEL
# ============================================================

print("\n")
print("=" * 70)
print("LOADING EMBEDDING MODEL")
print("=" * 70)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("Embedding model:", EMBEDDING_MODEL_NAME)



LOADING EMBEDDING MODEL


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model: all-MiniLM-L6-v2


### 12. Create document embeddings

Collects the chunk texts, then encodes them in one batched call.

`normalize_embeddings=True` scales every vector to unit length, which is what makes the inner-product index in the next cell equivalent to cosine similarity. The cast to `float32` is required: FAISS rejects `float64` input.

The printed matrix shape should read `(number_of_chunks, 384)`. If the first number disagrees with the chunk count from section 8, something was dropped.

In [12]:
# ============================================================
# 13. EXTRACT CHUNK TEXT
# ============================================================

chunk_texts = [
    chunk["text"]
    for chunk in chunks
]


# ============================================================
# 14. CREATE DOCUMENT EMBEDDINGS
# ============================================================

print("\n")
print("=" * 70)
print("CREATING DOCUMENT EMBEDDINGS")
print("=" * 70)

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# FAISS expects float32 vectors.
embeddings = embeddings.astype("float32")

print("\nEmbedding matrix shape:")
print(embeddings.shape)



CREATING DOCUMENT EMBEDDINGS


Batches:   0%|          | 0/8 [00:00<?, ?it/s]


Embedding matrix shape:
(235, 384)


### 13. Create the FAISS vector store and the chunk lookup

`IndexFlatIP` performs exhaustive inner-product search over every stored vector. On normalised vectors that is cosine similarity, and because it is exhaustive it returns exact nearest neighbours — no approximation error enters the measurements.

`index.ntotal` should match the embedding count.

The lookup that follows maps each vector's row position back to its chunk. FAISS returns integer positions, not content, so without this map the retrieved IDs cannot be resolved to text or page numbers.

In [13]:
# ============================================================
# 15. CREATE FAISS VECTOR STORE
# ============================================================

print("\n")
print("=" * 70)
print("CREATING FAISS VECTOR STORE")
print("=" * 70)

embedding_dimension = embeddings.shape[1]

# Inner Product on normalized vectors is equivalent
# to cosine similarity.
index = faiss.IndexFlatIP(
    embedding_dimension
)

index.add(embeddings)

print("Embedding dimension:", embedding_dimension)
print("Vectors stored:", index.ntotal)


# ============================================================
# 16. CREATE CHUNK LOOKUP
# ============================================================

# The embedding row index corresponds to the position
# of the chunk in the chunks list.

chunk_lookup = {
    position: chunk
    for position, chunk in enumerate(chunks)
}



CREATING FAISS VECTOR STORE
Embedding dimension: 384
Vectors stored: 235


### 14. Define the retrieval function

Embeds the query with the same model and normalisation used for the documents — a mismatch here would make the similarity scores meaningless — then searches the index.

`search_k = min(top_k, index.ntotal)` prevents requesting more neighbours than exist. FAISS marks unused slots with `-1`, which the loop skips.

Returns rank, chunk ID, page, similarity score and text per hit. **Rank is what MRR@3 is computed from, and the returned chunk IDs are what you compare against your ground-truth labels.**

In [14]:
# ============================================================
# 17. DEFINE RETRIEVAL FUNCTION
# ============================================================

def retrieve_documents(
    query,
    top_k=3
):
    """
    Retrieve the top matching chunks using FAISS.

    Returns chunk ID, page, similarity score, and text.
    """

    if not query or not query.strip():
        raise ValueError("The query cannot be empty.")

    if top_k <= 0:
        raise ValueError("top_k must be greater than 0.")

    # Convert the question into an embedding.
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    query_embedding = query_embedding.astype("float32")

    # Avoid requesting more results than available vectors.
    search_k = min(top_k, index.ntotal)

    # Search the FAISS vector store.
    scores, indices = index.search(
        query_embedding,
        search_k
    )

    results = []

    for rank, idx in enumerate(indices[0]):

        if idx == -1:
            continue

        chunk = chunk_lookup[int(idx)]

        results.append({
            "rank": rank + 1,
            "chunk_id": chunk["chunk_id"],
            "page": chunk["page"],
            "score": float(scores[0][rank]),
            "text": chunk["text"]
        })

    return results

### 15. Test retrieval

Prompts for a question and prints the top-K chunks with their similarity scores, or skips if the input is left empty.

Use this to run each of the three test questions and record which chunk IDs come back, in order. Those lists are the raw material for Precision@3, Recall@3 and MRR@3.

A high similarity score does not establish that a chunk contains the answer — slide 7 makes this point explicitly. Read the returned text, do not trust the number.

In [15]:
# ============================================================
# 18. TEST RETRIEVAL
# ============================================================

print("\n")
print("=" * 70)
print("TESTING VECTOR STORE")
print("=" * 70)

test_query = input(
    "\nEnter a test question "
    "(or press Enter to skip): "
)

if test_query.strip():

    test_results = retrieve_documents(
        test_query,
        TOP_K
    )

    print("\n")
    print("=" * 70)
    print("RETRIEVED CHUNKS")
    print("=" * 70)

    for result in test_results:

        print(f"\nRESULT {result['rank']}")
        print(f"Chunk ID   : {result['chunk_id']}")
        print(f"Page       : {result['page']}")
        print(f"Similarity : {result['score']:.4f}")

        print("-" * 70)

        print(result["text"])



TESTING VECTOR STORE

Enter a test question (or press Enter to skip): I'm considering Tala, but I've also seen a Tala Facebook page and another Tala-branded loan app. Is Tala in SEC records, and what warnings should I know about?


RETRIEVED CHUNKS

RESULT 1
Chunk ID   : 203
Page       : 111
Similarity : 0.6815
----------------------------------------------------------------------
C SEC regulatory events and advisories Record ID: EVT0033 Record date: 2026-06-09 Platform or entity: Tala-branded fraudulent OLPs/apps Company: Tala Financing Philippines Inc Evidence category: SEC_REGULATORY_EVENT_OR_ADVISORY Classification: IMPERSONATION_ADVISORY Evidence priority: NEWER_REGULATORY_EVIDENCE RAG source text (v5 normalized text) SEC regulatory evidence dated 2026-06-09. Platform/entity: Tala-branded fraudulent OLPs/apps. Company/operator: Tala Financing Philippines Inc. Classification: IMPERSONATION_ADVISORY. Source/event: Advisory against fraudulent online lending platforms and mobile app

### 16. Save the FAISS index

Serialises the index to `rag_faiss.index` so the vectors survive a runtime restart without re-embedding.

In [16]:
# ============================================================
# 19. SAVE FAISS INDEX
# ============================================================

FAISS_FILE = "rag_faiss.index"

faiss.write_index(
    index,
    FAISS_FILE
)

print("\nFAISS index saved:", FAISS_FILE)


FAISS index saved: rag_faiss.index


### 17. Save chunks and metadata

Pickles the chunk list, preserving each chunk's ID, page and text. The index stores only vectors, so without this file the retrieved positions cannot be resolved back to content.

In [17]:
# ============================================================
# 20. SAVE CHUNKS AND METADATA
# ============================================================

CHUNKS_FILE = "rag_chunks.pkl"

with open(CHUNKS_FILE, "wb") as f:

    pickle.dump(
        chunks,
        f
    )

print("Chunks saved:", CHUNKS_FILE)

Chunks saved: rag_chunks.pkl


### 18. Save the indexing configuration

Records the chunk size, overlap, top-K, embedding model and source filename alongside the index.

This is what keeps a set of saved artifacts interpretable after the fact. When you have several runs on disk, this file is how you tell which index came from which configuration.

In [18]:
# ============================================================
# 21. SAVE INDEXING CONFIGURATION
# ============================================================

CONFIG_FILE = "rag_config.pkl"

config = {
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "top_k": TOP_K,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "source_document": pdf_file
}

with open(CONFIG_FILE, "wb") as f:

    pickle.dump(
        config,
        f
    )

print("Configuration saved:", CONFIG_FILE)

Configuration saved: rag_config.pkl


### 19. Indexing complete

Prints a summary of the run and the files written. Capture this output for each configuration — it is the per-run record the comparison table is built from.

In [19]:
# ============================================================
# 22. INDEXING COMPLETE
# ============================================================

print("\n")
print("=" * 70)
print("INDEXING COMPLETE")
print("=" * 70)

print("Source document :", pdf_file)
print("Number of chunks:", len(chunks))
print("Vectors stored  :", index.ntotal)
print("Chunk size      :", CHUNK_SIZE)
print("Chunk overlap   :", CHUNK_OVERLAP)
print("Embedding model :", EMBEDDING_MODEL_NAME)

print("\nSaved files:")
print("  ", EXCEL_FILE)
print("  ", FAISS_FILE)
print("  ", CHUNKS_FILE)
print("  ", CONFIG_FILE)

print("\nThe vector store is ready for retrieval.")
print(
    "Inspect the downloaded Excel workbook to evaluate "
    "the generated chunks."
)



INDEXING COMPLETE
Source document : LendSuri_Member_3_Corpus_clean.pdf
Number of chunks: 235
Vectors stored  : 235
Chunk size      : 1000
Chunk overlap   : 200
Embedding model : all-MiniLM-L6-v2

Saved files:
   LendSuri_Member_3_Corpus_clean_chunks_inspection.xlsx
   rag_faiss.index
   rag_chunks.pkl
   rag_config.pkl

The vector store is ready for retrieval.
Inspect the downloaded Excel workbook to evaluate the generated chunks.


# Component II. Prompt and Retrieval

### 1. Import libraries for generation

**Changed for Step 3.1.** The notebook shipped with `AutoModelForSeq2SeqLM`, which is correct for FLAN-T5 (an encoder-decoder model) but cannot load the assigned model. Llama-SEA-LION-v3-8B-IT is decoder-only, so it needs `AutoModelForCausalLM` — loading it with the seq2seq class raises an error outright.

`BitsAndBytesConfig` and the two installed packages provide 4-bit quantization, without which an 8B model does not fit the free-tier T4. `torch` handles device placement and inference control.

In [ ]:
# ============================================================
# RAG PROMPT PROCESSING
# TOKEN-LIMITED CONTEXT, ANSWER GENERATION, AND DIAGNOSTICS
# Model: aisingapore/Llama-SEA-LION-v3-8B-IT
# ============================================================


# ============================================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================================

# CHANGED (Step 3.1): 4-bit quantization support for the assigned 8B model.
!pip install -q bitsandbytes accelerate

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,      # CHANGED: SEA-LION is decoder-only, not seq2seq
    BitsAndBytesConfig         # CHANGED: enables 4-bit loading
)

import torch
import re

### 2. Check the device

Selects CUDA when a GPU is present and falls back to CPU. FLAN-T5-base ran acceptably on CPU, but the assigned SEA-LION model is loaded in 4-bit with `bitsandbytes`, which needs CUDA — this section must print `Device: cuda` before section 3 is run.

In [ ]:
# ============================================================
# 2. CHECK DEVICE
# ============================================================

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 70)
print("RAG PROMPT PROCESSING")
print("=" * 70)

print("Device:", device)

### 3. Load the language model

**Changed for Step 3.1.** `LLM_NAME` is now the group's assigned model, `aisingapore/Llama-SEA-LION-v3-8B-IT`, replacing the notebook's baseline `google/flan-t5-base`. It is a Llama 3.1-architecture decoder model, 8B parameters, ungated under the Llama 3.1 Community License.

Two consequences follow from that swap.

**Quantization.** The published weights are BF16, roughly 16 GB, against the free-tier T4's ~15 GB. Loading in 4-bit NF4 brings the resident model to about 5.5 GB. The download is still the full 16 GB, so expect several minutes on the first run. This cell requires a GPU runtime — confirm section 2 printed `Device: cuda` before running it.

**Context budget.** `MAX_INPUT_TOKENS` rises from 512 to 4096. The 512 figure was FLAN-T5's encoder ceiling, and at `CHUNK_SIZE = 1800` the prompt builder would have discarded most of the retrieved evidence to respect a limit this model does not have. SEA-LION's native context is 128K; 4096 is simply a comfortable budget for three chunks of 1800 characters.

`.to(device)` is removed because `device_map="auto"` already places a quantized model, and calling it raises.

In [ ]:
# ============================================================
# 3. LOAD THE LOCAL LLM
# ============================================================

# CHANGED (Step 3.1): the group's assigned language model, replacing the
# notebook's baseline google/flan-t5-base.
LLM_NAME = "aisingapore/Llama-SEA-LION-v3-8B-IT"

# CHANGED: SEA-LION supports a 128K context window. 4096 is ample for
# TOP_K = 3 at CHUNK_SIZE = 1800 and removes flan-t5's 512-token encoder
# limit, which would otherwise discard retrieved evidence.
MAX_INPUT_TOKENS = 4096

# CHANGED (Step 3.2, revision 2): 32 truncated badly; 128 still cut the
# two-part Q2 answer off mid-word. 256 completes it.
MAX_NEW_TOKENS = 256

# Reserve space within the input budget for safety.
RESERVED_TOKEN_MARGIN = 16

print("\nLoading language model:")
print(LLM_NAME)

# CHANGED: 4-bit NF4 quantization. Weights are published in BF16 (~16 GB);
# quantizing on load brings the resident model to roughly 5.5 GB, which fits
# the free-tier T4's ~15 GB.
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    LLM_NAME
)

llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    quantization_config=quant_config,
    device_map="auto"
)

# CHANGED: device_map="auto" has already placed the model. Calling
# .to(device) on a quantized model raises an error.
llm.eval()

print("Language model loaded.")

### 4. Define the token counting function

Wraps the tokenizer to return an exact token count, with `truncation=False` so that an over-long prompt is measured honestly rather than silently cut. Used by the prompt builder to verify its own output before generation.

In [ ]:
# ============================================================
# 4. DEFINE TOKEN COUNTING FUNCTION
# ============================================================

def count_tokens(text):

    return len(
        tokenizer.encode(
            text,
            add_special_tokens=True,
            truncation=False
        )
    )

### 5. Prioritise retrieved documents

Sorts the retrieved chunks by descending similarity score, preserving FAISS's ranking. Because the index uses inner product over normalised vectors, higher means more similar.

Ordering matters: the prompt builder fills its token budget from the top down, so the least similar chunk is the first to be dropped.

In [ ]:
# ============================================================
# 5. PRIORITIZE RETRIEVED DOCUMENTS
# ============================================================

def prioritize_documents(question, retrieved_documents):
    """
    Preserve FAISS similarity ranking.

    Assumes higher scores mean greater similarity,
    as with normalized embeddings and IndexFlatIP.
    """

    return sorted(
        retrieved_documents,
        key=lambda result: result.get("score", float("-inf")),
        reverse=True
    )

### 6. Build the token-limited RAG prompt

Assembles the prompt within the input budget. The question and instruction are tokenised first and their cost subtracted; whatever remains is the context budget. Chunks are added in rank order, each tagged with its page and chunk ID. A chunk that does not fit whole is cut at a token boundary and the loop then stops.

**Changed for this dataset.** The instruction text from Part 1 targeted capital figures and deadlines in a different document. It is rewritten for the LendSuri records, keeping the grounding guardrails — answer only from supplied evidence, never invent, refuse explicitly when evidence is missing — and adding the rules this corpus needs: every record is dated, a registry listing is not proof of current authorization or of a page's authenticity, and an advisory about unauthorized use of a company's name concerns the impersonator, not the registered company.

The assembled prompt is wrapped in SEA-LION's chat template via `apply_chat_template`, so the instruction-tuned model answers rather than continuing the text.

Component III measures retrieval only, so this prompt does not affect any Part 2 metric.

In [ ]:
# ============================================================
# 6. DEFINE THE TOKEN-LIMITED RAG PROMPT
# ============================================================

def create_rag_prompt(
    question,
    retrieved_documents
):
    """
    Build a prompt that preserves the question and includes
    as much relevant retrieved evidence as the input budget
    allows.

    Context is trimmed only at token boundaries.
    """

    ranked_documents = prioritize_documents(
        question,
        retrieved_documents
    )

    #instruction = (
    #    "Answer the question using only the evidence below. "
    #    "Give a concise answer. Do not invent information. "
    #    'If the evidence does not support an answer, say: '
    #    '"I cannot find the answer in the provided document."'
    #)

    # CHANGED FOR THIS DATASET: rewritten for the LendSuri Member 3 records
    # (dated registry entries vs. later SEC advisories).
    instruction = (
        "Answer the question using only the supplied evidence records. "
        "Every record is dated: give the date and the record type "
        "(2026 registry snapshot, 2021 historical list, SEC advisory, "
        "framework guidance or complaint guidance) for each fact you use. "
        "Write platform and company names exactly as they appear. "
        "A registry listing only shows that a platform was recorded on "
        "that date; it does not prove current authorization and does not "
        "show that any Facebook page, app or website is genuine. "
        "An advisory about unauthorized use of a company's name concerns "
        "the impersonator, not the registered company, so do not "
        "attribute it to the company. Answer each part of a multi-part "
        "question separately and name any part the evidence does not "
        "cover. Do not guess or use outside knowledge. If the evidence "
        "does not establish the answer, respond exactly: "
        "'I cannot find the answer in the provided document.'"
    )

    # Keep the question near the beginning.
    prompt_prefix = (
        f"Question: {question}\n\n"
        f"{instruction}\n\n"
        "Evidence:\n"
    )

    prompt_suffix = "\n\nAnswer:"

    # Tokenize the fixed parts first.
    prefix_tokens = tokenizer.encode(
        prompt_prefix,
        add_special_tokens=False
    )

    suffix_tokens = tokenizer.encode(
        prompt_suffix,
        add_special_tokens=False
    )

    # Keep the final prompt safely within the model limit.
    available_context_tokens = (
        MAX_INPUT_TOKENS
        - len(prefix_tokens)
        - len(suffix_tokens)
        - RESERVED_TOKEN_MARGIN
        - 1  # Account for a possible special token.
    )

    if available_context_tokens <= 0:
        raise ValueError(
            "The question and instructions leave no room "
            "for retrieved evidence. Shorten the question."
        )

    selected_parts = []
    remaining_tokens = available_context_tokens

    for result in ranked_documents:

        if remaining_tokens <= 0:
            break

        source_text = (
            f"[Page {result['page']}, "
            f"Chunk {result['chunk_id']}]\n"
            f"{result['text']}"
        )

        # Tokenize each source separately.
        source_tokens = tokenizer.encode(
            source_text,
            add_special_tokens=False
        )

        separator_tokens = (
            tokenizer.encode(
                "\n\n",
                add_special_tokens=False
            )
            if selected_parts
            else []
        )

        tokens_available = (
            remaining_tokens - len(separator_tokens)
        )

        if tokens_available <= 0:
            break

        # Include the full chunk when it fits.
        # Otherwise, retain the beginning of the chunk.
        included_tokens = source_tokens[
            :tokens_available
        ]

        if not included_tokens:
            continue

        included_text = tokenizer.decode(
            included_tokens,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        selected_parts.append(included_text)

        remaining_tokens -= (
            len(separator_tokens)
            + len(included_tokens)
        )

        # Stop if this chunk had to be truncated.
        if len(included_tokens) < len(source_tokens):
            break

    context = "\n\n".join(selected_parts)

    prompt = (
        prompt_prefix
        + context
        + prompt_suffix
    )

    # CHANGED (Step 3.2): SEA-LION is instruction-tuned and expects its chat
    # template. Without it the model continues the text rather than answering.
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True
    )

    # Final check using the actual tokenizer.
    final_token_count = count_tokens(prompt)

    if final_token_count > MAX_INPUT_TOKENS:
        raise ValueError(
            f"Prompt contains {final_token_count} tokens, "
            f"exceeding the {MAX_INPUT_TOKENS}-token limit. "
            "Reduce the context budget."
        )

    print("\n--- PROMPT DIAGNOSTICS ---")
    print("Original question:", question)
    print("Retrieved chunks:", len(retrieved_documents))
    print("Chunks included:", len(selected_parts))
    print("Final prompt tokens:", final_token_count)
    print("Input token limit:", MAX_INPUT_TOKENS)

    print("\n--- SELECTED EVIDENCE ---")

    for i, part in enumerate(selected_parts, start=1):
      print(f"\nEvidence {i}:")
      print(part)

    return prompt

### 7. Define the answer generation function

Tokenises the prompt with `truncation=False` and raises if it exceeds the limit, so silent truncation cannot corrupt a result. Generation runs under `torch.no_grad()` with `do_sample=False`, making output deterministic — the same prompt yields the same answer every run, which is what makes configurations comparable.

**Three changes for the assigned model.**

Inputs move to `llm.device` rather than the `device` variable, because `device_map="auto"` decides placement for a quantized model.

Beam search drops from four beams to greedy. On an 8B model at 4-bit this is roughly a 4x speedup, and beam search buys little on short factual answers. Determinism comes from `do_sample=False`, not from beams, so comparability is unaffected. `pad_token_id` is set because Llama tokenizers define no pad token.

The decode now slices `outputs[0][input_length:]`. A causal LM returns the prompt followed by its continuation in one sequence; without the slice the "answer" would be the prompt echoed back. FLAN-T5, as an encoder-decoder model, returned only the generated text, which is why the original needed no slice.

In [ ]:
# ============================================================
# 7. DEFINE ANSWER GENERATION FUNCTION
# ============================================================

def generate_answer(
    prompt,
    max_new_tokens=MAX_NEW_TOKENS
):
    """
    Generate an answer without silently truncating the prompt.
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=False
    )

    input_length = inputs["input_ids"].shape[1]

    print("\n--- GENERATION DIAGNOSTICS ---")
    print("Actual input tokens:", input_length)
    print("Maximum input tokens:", MAX_INPUT_TOKENS)

    if input_length > MAX_INPUT_TOKENS:
        raise ValueError(
            f"Input contains {input_length} tokens, "
            f"but the model limit is {MAX_INPUT_TOKENS}."
        )

    # CHANGED: the quantized model sets its own device placement, so inputs
    # go to wherever the model actually landed.
    inputs = {
        key: value.to(llm.device)
        for key, value in inputs.items()
    }

    # Generate deterministically.
    with torch.no_grad():

        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            # CHANGED: greedy. Four-beam search on an 8B 4-bit model costs
            # roughly 4x the time for no benefit on short factual answers.
            # Determinism comes from do_sample=False, not from beams.
            num_beams=1,
            # CHANGED: Llama tokenizers define no pad token.
            pad_token_id=tokenizer.eos_token_id
        )

    # CHANGED: a causal LM returns the prompt followed by its continuation,
    # so the prompt tokens are sliced off. FLAN-T5, being encoder-decoder,
    # returned the generated answer alone.
    answer = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    ).strip()

    if not answer:
        return (
            "The model generated an empty answer. "
            "Inspect the prompt and retrieved evidence."
        )

    return answer

### 8. Define the complete RAG query function

Chains the pieces: validate the question, retrieve, build the prompt, generate, and return the answer with its sources and the prompt itself.

Returning the prompt is what makes a wrong answer diagnosable — you can see whether the gold evidence reached the model at all, which separates a retrieval failure from a generation failure.

In [ ]:
# ============================================================
# 8. DEFINE COMPLETE RAG QUERY FUNCTION
# ============================================================


def rag_query(question, top_k=3):
    """
    General-purpose RAG question-answering function.

    Workflow:
    1. Retrieve relevant document chunks.
    2. Build a token-limited prompt.
    3. Generate an answer using the language model.
    4. Return the answer, retrieved sources, and prompt.
    """

    # STEP 1: Validate the question.
    question = question.strip()

    if not question:
        return (
            "Please enter a valid question.",
            [],
            ""
        )

    # STEP 2: Retrieve relevant document chunks.
    retrieved_documents = retrieve_documents(
        question,
        top_k=top_k
    )

    # STEP 3: Handle cases where no evidence is retrieved.
    if not retrieved_documents:
        return (
            "I cannot find the answer in the provided document.",
            [],
            ""
        )

    # STEP 4: Construct the token-limited RAG prompt.
    prompt = create_rag_prompt(
        question,
        retrieved_documents
    )

    # STEP 5: Generate an answer from the retrieved evidence.
    answer = generate_answer(prompt)

    # STEP 6: Handle an empty model response.
    if not answer or not answer.strip():
        answer = (
            "The model could not generate an answer. "
            "Please review the retrieved evidence."
        )

    # STEP 7: Return the results.
    return (
        answer.strip(),
        retrieved_documents,
        prompt
    )

### 9. Continuous question-answer loop

Accepts questions until you exit, printing the answer and its supporting chunks each time.

Run all three test questions here for every configuration, and record both the answer and the supporting chunk IDs. The chunk IDs feed the retrieval metrics; the answers feed the correctness judgement.

In [ ]:
# ============================================================
# 9. CONTINUOUS QUESTION-ANSWER LOOP
# ============================================================

print("\n")
print("=" * 70)
print("CONTINUOUS RAG QUESTION ANSWERING")
print("=" * 70)

print("Ask questions about the indexed document.")
print("Type 'exit' to terminate.")
print("Type 'sources' to display sources from the last query.")
print("Type 'prompt' to display the last generated prompt.")


# Store the most recent query results.
last_answer = None
last_sources = []
last_prompt = None


while True:

    print("\n")

    user_question = input(
        "Question: "
    ).strip()

    # --------------------------------------------------------
    # EXIT CONDITION
    # --------------------------------------------------------

    if user_question.lower() == "exit":

        print("\nRAG session terminated.")
        break

    # --------------------------------------------------------
    # IGNORE EMPTY QUESTIONS
    # --------------------------------------------------------

    if not user_question:

        print("Please enter a question.")
        continue

    # --------------------------------------------------------
    # SPECIAL COMMAND: SOURCES
    # --------------------------------------------------------

    if user_question.lower() == "sources":

        if not last_sources:
            print("No previous query sources are available.")
            continue

        print("\n" + "=" * 70)
        print("SOURCES FROM LAST QUERY")
        print("=" * 70)

        for result in last_sources:

            print(
                f"\nRank {result['rank']} | "
                f"Page {result['page']} | "
                f"Chunk {result['chunk_id']} | "
                f"Similarity {result['score']:.4f}"
            )

            print("-" * 70)
            print(result["text"])

        continue

    # --------------------------------------------------------
    # SPECIAL COMMAND: PROMPT
    # --------------------------------------------------------

    if user_question.lower() == "prompt":

        if last_prompt is None:
            print("No previous query prompt is available.")
            continue

        print("\n" + "=" * 70)
        print("LAST RAG PROMPT")
        print("=" * 70)

        print(last_prompt)

        continue

    # --------------------------------------------------------
    # PROCESS QUESTION
    # --------------------------------------------------------

    try:

        answer, retrieved_documents, prompt = rag_query(
            user_question,
            top_k=TOP_K
        )

        last_answer = answer
        last_sources = retrieved_documents
        last_prompt = prompt

        # ----------------------------------------------------
        # DISPLAY ANSWER
        # ----------------------------------------------------

        print("\n")
        print("=" * 70)
        print("RAG ANSWER")
        print("=" * 70)

        print(answer)

        # ----------------------------------------------------
        # DISPLAY RETRIEVED SOURCES
        # ----------------------------------------------------

        print("\n")
        print("=" * 70)
        print("RETRIEVED SOURCES")
        print("=" * 70)

        for result in retrieved_documents:

            print(
                f"\n[{result['rank']}] "
                f"Page {result['page']} "
                f"| Chunk {result['chunk_id']} "
                f"| Similarity {result['score']:.4f}"
            )

            print("-" * 70)
            print(result["text"])

        # ----------------------------------------------------
        # DISPLAY PROMPT
        # ----------------------------------------------------

        print("\n")
        print("=" * 70)
        print("RAG PROMPT SENT TO LLM")
        print("=" * 70)

        print(prompt)

    except Exception as e:

        print("\nERROR:")
        print(str(e))

        print("\nThe question could not be processed.")


# ============================================================
# 10. END
# ============================================================

print("\n")
print("=" * 70)
print("RAG PROGRAM FINISHED")
print("=" * 70)

# Component III. Part 2 — Chunking Experiment (PT-F2)

**Student:** Audrey Benjamin Gutierrez · Section 401 I · Group QuantumStack, Member 3
**Course:** ITC-C508 — Deep Learning (NLP)

---

## What to run

This component is a **retrieval** experiment: Steps 3 and 5 of the brief measure Precision@3, Recall@3 and MRR@3, none of which involve the language model. Component II therefore does **not** need to be run, and the 16 GB SEA-LION download can be skipped. The language model and prompt remain the fixed settings recorded in Component II.

Run, in order: **Component I sections 1–19**, then the two cells below. Section 5 uploads `LendSuri_Member_3_Corpus_clean.pdf`; section 6 produces `pages`; section 11 loads the embedding model.

## Experimental controls

Held constant across all twelve configurations, per Step 1 of the brief:

| Parameter | Setting |
| --- | --- |
| Dataset | `LendSuri_Member_3_Corpus_clean.pdf` — the same file for every configuration, 126 pages |
| Embedding model | `all-MiniLM-L6-v2`, 384 dimensions |
| Vector store | FAISS `IndexFlatIP` over L2-normalised vectors (exact cosine search) |
| Retrieved chunks (k) | 3 |
| Test questions | The same nine questions, in the same order, for every configuration |
| Relevance criteria | Identical evidence-unit definitions for every configuration |

Each configuration builds a **new** index from its own chunks. No index is reused across configurations.

## The twelve configurations

A 4 × 3 factorial: four chunk sizes at 0%, 10% and 20% overlap. C1–C5 are the combinations given in the brief; C6–C12 complete the design. C9 (1000 / 200) is the baseline. A cleaned Member 3 page is at most 1,444 characters (about 1,030 on average), so 1800-character chunks hold one whole page each.

| | 0% overlap | 10% overlap | 20% overlap |
| --- | --- | --- | --- |
| **300 chars** | C1 — 0 | C2 — 30 | C3 — 60 |
| **500 chars** | C4 — 0 | C5 — 50 | C6 — 100 |
| **1000 chars** | C7 — 0 | C8 — 100 | C9 — 200 *(baseline)* |
| **1800 chars** | C10 — 0 | C11 — 180 | C12 — 360 |

Expressing overlap as a ratio is what makes Step 7 question 5 — comparing chunk sizes *at the same overlap percentage* — answerable.

## How the metrics are defined

**Evidence units.** Each question's evidence is a list of exact answer-bearing passages on its gold pages (18 units across the nine questions). A unit's position is found by searching for its text in the normalised page; the cell stops if a passage is missing or occurs more than once.

**Relevance.** A retrieved chunk counts as supporting an evidence unit only when it **contains that unit in full**. The brief requires that a relevant chunk "contains evidence that supports answering the question" and that "merely mentioning the same topic or keywords is not sufficient". This matters here because every page shares the same boilerplate: a chunk from the right page that holds only template text names no platform, company or advisory and supports nothing.

**Precision@3** — relevant chunks among the top three, divided by three. For a single-unit question, Precision@3 is at most 0.333 unless overlapping chunks each hold the whole passage, so that two or three retrieved chunks count as relevant (this happens once: C6, E2 = 0.667). Read the easy-question precision with that in mind.

**Recall@3** — distinct evidence units covered by the top three chunks, divided by the units required for that question. The denominator is fixed per question, so it does not drift with chunk size. A unit found in two retrieved chunks counts once.

**Max Recall@3** — the share of a question's evidence units that *any* chunk in that configuration holds in full (capped at three units). Below 1.000 the configuration cannot reach some evidence at all, because the passage straddles a chunk boundary.

**MRR@3** — the mean, across all nine questions, of 1/rank of the first relevant chunk (0 if none appears in the top three). Per Step 5D the overall figure is taken across all questions, never by averaging the three difficulty-level means.

### Cell 1 — run the twelve configurations

First locates every evidence passage in the normalised page text and checks it occurs exactly once. Then loops over the configurations, and for each one: chunks the same source document, counts the chunks, embeds them with the same model, builds a fresh FAISS index, runs all nine questions at k=3, and records every retrieved chunk with its rank, source reference, similarity, relevance verdict and the evidence units it covers.

Gold chunks are **derived inside the loop** from the located character spans rather than hard-coded, so the mapping is recomputed for every configuration. The assertion on the first line of the loop enforces the brief's requirement that overlap be smaller than chunk size.

Embedding time, index build time, mean query latency and index size are captured for the efficiency comparison in Step 8.

In [20]:
# ============================================================
# PART 2 (PT-F2): AUTOMATED CHUNKING EXPERIMENT
# 12 configurations x 9 questions, one fresh FAISS index per configuration
# ============================================================

import time
import pandas as pd

# ------------------------------------------------------------
# A. The 12 configurations: 4 chunk sizes x 3 overlap ratios (0%, 10%, 20%)
# ------------------------------------------------------------

CONFIGS = [
    ("C1",   300,   0), ("C2",   300,  30), ("C3",   300,  60),
    ("C4",   500,   0), ("C5",   500,  50), ("C6",   500, 100),
    ("C7",  1000,   0), ("C8",  1000, 100), ("C9",  1000, 200),   # C9 = Part 1 baseline
    ("C10", 1800,   0), ("C11", 1800, 180), ("C12", 1800, 360),
]

EXPERIMENT_TOP_K = 3

# ------------------------------------------------------------
# B. The 9 test questions and their verified evidence units.
#
#    Each evidence unit is (label, page, passage): the exact answer-bearing
#    text on that page. Positions are located below in the whitespace-
#    normalised page text produced by section 6, so gold chunks are
#    re-derived for every configuration rather than hard-coded.
# ------------------------------------------------------------

QUESTION_SPECS = {'M3-E': {'difficulty': 'Easy',
          'text': 'Which lending company is Peso2you listed under in the SEC records?',
          'answer': 'The January 26, 2026 snapshot associates Peso2you with Snapcash Lending, Inc.',
          'evidence': [('2026 registry - Peso2you under Snapcash',
                        46,
                        'Platform: Peso2you. Company: Snapcash Lending, Inc.')]},
 'E2': {'difficulty': 'Easy',
        'text': "Which company is the Atome PH platform recorded under in the SEC's January 2026 "
                'registry snapshot?',
        'answer': 'Neuroncredit Financing Company Inc.',
        'evidence': [('2026 registry - Atome PH under Neuroncredit',
                      35,
                      'Platform: Atome PH. Company: Neuroncredit Financing Company Inc.')]},
 'E3': {'difficulty': 'Easy',
        'text': 'Which SEC department should receive a complaint about a company lending without a '
                'Certificate of Authority?',
        'answer': 'The SEC Enforcement and Investor Protection Department (EIPD).',
        'evidence': [('complaint channel - EIPD',
                      126,
                      'Primary SEC channel: SEC Enforcement and Investor Protection Department '
                      '(EIPD).')]},
 'M3-A': {'difficulty': 'Average',
          'text': 'BillEase appears under First Digital Finance in SEC records, but I saw another '
                  'lender using that company name. Has the SEC warned about this?',
          'answer': 'The January 2026 snapshot lists BillEase under First Digital Finance '
                    'Corporation. A May 21, 2026 advisory entry concerns an unregistered lender or '
                    "unrecorded OLP using the company's identity. The advisory must not "
                    'automatically be attributed to BillEase itself.',
          'evidence': [('2026 registry - Billease under First Digital',
                        15,
                        'Platform: Billease. Company: First Digital Finance Corporation.'),
                       ('2026-05-21 advisory - unauthorized use of First Digital name',
                        109,
                        'Source/event: Public advisory on unauthorized use of First Digital '
                        'Finance Corporation by an unregistered lending company and unrecorded '
                        'OLP.')]},
 'A2': {'difficulty': 'Average',
        'text': 'Mabilis Cash appears in both an older and a newer SEC list. Which company is it '
                'recorded under, and what are the dates of those two lists?',
        'answer': 'Mali Lending Corp, in both the November 29, 2021 historical list and the '
                  'January 26, 2026 registry snapshot.',
        'evidence': [('2021 list - Mabilis Cash under Mali Lending',
                      82,
                      'Company: Mali Lending Corp. SEC registration no.: CS201907662. Certificate '
                      'of Authority no.: 2968. Recorded OLP/platform names: Mabilis Cash.'),
                     ('2026 registry - Mabilis Cash under Mali Lending',
                      32,
                      'SEC OLP current-registry snapshot dated 2026-01-26. Platform: Mabilis Cash. '
                      'Company: Mali Lending Corp.')]},
 'A3': {'difficulty': 'Average',
        'text': 'What did the SEC order financing and lending companies with online lending '
                'platforms to have in June 2025, and can that order be used as proof of a '
                "platform's status?",
        'answer': 'A June 20, 2025 order requires financing and lending companies with OLPs/OLAs '
                  'to have landline numbers. It is operational-requirement context only and is not '
                  "proof of a platform's status.",
        'evidence': [('2025-06-20 order - landline numbers',
                      124,
                      'Order to All Financing and Lending Companies with OLPs/OLAs to Have '
                      'Landline Numbers.'),
                     ('status-use rule - not platform-status proof',
                      124,
                      'Status-use rule: Operational requirement context only; not platform-status '
                      'proof.')]},
 'M3-D': {'difficulty': 'Difficult',
          'text': "I'm considering Tala, but I've also seen a Tala Facebook page and another "
                  'Tala-branded loan app. Is Tala in SEC records, and what warnings should I know '
                  'about?',
          'answer': 'The January 2026 registry snapshot includes Tala under Tala Financing '
                    'Philippines Inc. A February 12, 2026 advisory entry concerns an unauthorized '
                    'Facebook page impersonating Tala, and a June 9, 2026 advisory entry concerns '
                    'fraudulent OLPs/mobile apps misusing its name. The dated listing does not '
                    'authenticate a particular page or app.',
          'evidence': [('2026 registry - Tala',
                        54,
                        'Platform: Tala. Company: Tala Financing Philippines Inc.'),
                       ('2026-02-12 advisory - fake Tala Facebook page',
                        105,
                        'Source/event: Public advisory on an unauthorized Facebook page '
                        'impersonating Tala Financing Philippines.'),
                       ('2026-06-09 advisory - fraudulent Tala apps',
                        111,
                        'Source/event: Advisory against fraudulent online lending platforms and '
                        'mobile applications illegally using the name of Tala Financing '
                        'Philippines Inc.')]},
 'D2': {'difficulty': 'Difficult',
        'text': 'ACOM - Fast Cash Loan appears in SEC lists, but I saw Facebook pages using the '
                'ACOM name. Which company operates ACOM, is it in both the 2021 and January 2026 '
                'lists, and has the SEC warned about fake ACOM pages?',
        'answer': 'Acom Consumer Finance Corporation. It appears in the November 29, 2021 '
                  'historical list and the January 26, 2026 snapshot. A July 22, 2026 SEC advisory '
                  'concerns fake Facebook pages using the ACOM Consumer Finance Corporation name; '
                  'the listings do not authenticate any Facebook page.',
        'evidence': [('2021 list - ACOM under Acom Consumer Finance',
                      61,
                      'Company: Acom Consumer Finance Corporation. SEC registration no.: '
                      'CS201724897. Certificate of Authority no.: 1140. Recorded OLP/platform '
                      'names: ACOM - Fast Cash Loan. Simple.'),
                     ('2026 registry - ACOM under Acom Consumer Finance',
                      1,
                      'Platform: ACOM – Fast Cash Loan. Simple.. Company: Acom Consumer Finance '
                      'Corporation.'),
                     ('2026-07-22 advisory - fake ACOM Facebook pages',
                      119,
                      'Source/event: Public advisory on unauthorized use of ACOM Consumer Finance '
                      'Corporation name by fake Facebook pages.')]},
 'D3': {'difficulty': 'Difficult',
        'text': 'Snapcash Lending recorded several platforms in the 2021 SEC list. Which platforms '
                'were they, and are Peso2you and Jetpeso still listed under Snapcash in the '
                'January 2026 snapshot?',
        'answer': 'In the November 29, 2021 list Snapcash Lending recorded Masayacash, Peso2you, '
                  'CashniJuan, Peso2go and Jetpeso. Both Peso2you and Jetpeso appear under '
                  'Snapcash Lending, Inc. in the January 26, 2026 snapshot.',
        'evidence': [('2021 list - Snapcash platforms',
                      95,
                      'Recorded OLP/platform names: Masayacash, Peso2you, CashniJuan, Peso2go, '
                      'Jetpeso.'),
                     ('2026 registry - Peso2you under Snapcash',
                      46,
                      'Platform: Peso2you. Company: Snapcash Lending, Inc.'),
                     ('2026 registry - Jetpeso under Snapcash',
                      49,
                      'Platform: Jetpeso. Company: Snapcash Lending, Inc.')]}}

# Locate every passage and convert it to (label, page, char_start, char_end).
page_text_by_number = {p["page"]: p["text"] for p in pages}
QUESTIONS = {}
for qid, spec in QUESTION_SPECS.items():
    units = []
    for label, page, passage in spec["evidence"]:
        text = page_text_by_number[page]
        hits = text.count(passage)
        assert hits == 1, f"{qid} '{label}': found {hits} times on page {page}"
        start = text.find(passage)
        units.append((label, page, start, start + len(passage)))
    QUESTIONS[qid] = {**spec, "evidence": units}

print("Evidence units located:", sum(len(q["evidence"]) for q in QUESTIONS.values()),
      "across", len(QUESTIONS), "questions (each found exactly once)")

# ------------------------------------------------------------
# C. Run the experiment
# ------------------------------------------------------------

def gold_units_for(chunk_list, question):
    """Map each evidence unit to the chunk IDs that CONTAIN IT IN FULL.

    Strict containment, not mere overlap. The brief requires that a relevant
    chunk "contains evidence that supports answering the question" and that
    "merely mentioning the same topic or keywords is not sufficient" -- a chunk
    holding a five-character fragment of a 116-character provision supports
    nothing. Overlap-based matching also biases the comparison: it inflates
    small-chunk scores, which produce many partial hits, while leaving
    1800-character chunks untouched, because a large chunk either holds a unit
    whole or not at all.
    """
    out = {}
    for label, page, start, end in question["evidence"]:
        out[label] = {
            c["chunk_id"] for c in chunk_list
            if c["page"] == page and c["start"] <= start and c["end"] >= end
        }
    return out


def create_chunks_indexed(pages, chunk_size, chunk_overlap):
    """Same algorithm as section 7, but also records each chunk's character span."""
    out = []
    cid = 0
    step = chunk_size - chunk_overlap
    for page_data in pages:
        text = page_data["text"]
        start = 0
        while start < len(text):
            raw = text[start:start + chunk_size]
            if raw.strip():
                out.append({
                    "chunk_id": cid,
                    "page": page_data["page"],
                    "start": start,
                    "end": start + len(raw),
                    "text": raw.strip(),
                })
                cid += 1
            start += step
    return out


detail_rows = []
config_rows = []

print("=" * 70)
print("PT-F2 CHUNKING EXPERIMENT")
print("=" * 70)

for cfg, cs, ov in CONFIGS:

    assert ov < cs, f"{cfg}: overlap must be smaller than chunk size"

    # 1-3. chunk the same source document and count the chunks
    ch = create_chunks_indexed(pages, cs, ov)

    # 4. embed with the same model
    t0 = time.time()
    emb = embedding_model.encode(
        [c["text"] for c in ch],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype("float32")
    embed_s = time.time() - t0

    # 5. build a NEW index for this configuration
    t1 = time.time()
    idx = faiss.IndexFlatIP(emb.shape[1])
    idx.add(emb)
    index_s = time.time() - t1

    lookup = {i: c for i, c in enumerate(ch)}

    # 6-9. run every question against this index
    q_lat = []
    for qid, q in QUESTIONS.items():

        gold = gold_units_for(ch, q)
        gold_any = set().union(*gold.values()) if gold else set()

        # How many of this question's units ANY chunk in this configuration can
        # supply. Units below this count are unreachable at this chunk size and
        # overlap: they straddle a boundary and no single chunk holds them whole,
        # so retrieval cannot recover them however good it is.
        reachable = sum(1 for s in gold.values() if s)

        t2 = time.time()
        qv = embedding_model.encode(
            [q["text"]], convert_to_numpy=True, normalize_embeddings=True
        ).astype("float32")
        scores, ids = idx.search(qv, min(EXPERIMENT_TOP_K, idx.ntotal))
        q_lat.append(time.time() - t2)

        for rank, cid in enumerate(ids[0], start=1):
            if cid == -1:
                continue
            c = lookup[int(cid)]
            covered = sorted(lab for lab, s in gold.items() if int(cid) in s)
            detail_rows.append({
                "Configuration": cfg,
                "Chunk size": cs,
                "Overlap": ov,
                "Question ID": qid,
                "Difficulty": q["difficulty"],
                "Expected evidence": "; ".join(f"{lab} (p.{pg})" for lab, pg, _, _ in q["evidence"]),
                "Evidence units required": len(q["evidence"]),
                "Evidence units reachable": reachable,
                "Retrieved rank": rank,
                "Retrieved chunk ID": int(cid),
                "Source reference": f"Page {c['page']}, chars {c['start']}-{c['end']}",
                "Similarity": round(float(scores[0][rank - 1]), 4),
                "Relevant?": "Yes" if int(cid) in gold_any else "No",
                "Evidence covered": "; ".join(covered),
                "Chunk text": c["text"][:400],
            })

    config_rows.append({
        "Configuration": cfg,
        "Chunk size": cs,
        "Overlap": ov,
        "Overlap %": round(100 * ov / cs),
        "Total chunks": len(ch),
        "Mean chunk chars": round(sum(len(c["text"]) for c in ch) / len(ch), 1),
        "Embedding time (s)": round(embed_s, 2),
        "Index build time (s)": round(index_s, 3),
        "Mean query latency (ms)": round(1000 * sum(q_lat) / len(q_lat), 2),
        "Index size (KB)": round(emb.nbytes / 1024, 1),
    })

    print(f"{cfg:<4} size={cs:<5} overlap={ov:<4} chunks={len(ch):<5} "
          f"embed={embed_s:5.1f}s  query={1000*sum(q_lat)/len(q_lat):5.1f}ms")

detail_df = pd.DataFrame(detail_rows)
config_df = pd.DataFrame(config_rows)

print("\nDetailed retrieval log rows:", len(detail_df),
      f"(expected {len(CONFIGS)} x {len(QUESTIONS)} x {EXPERIMENT_TOP_K} = "
      f"{len(CONFIGS) * len(QUESTIONS) * EXPERIMENT_TOP_K})")

Evidence units located: 18 across 9 questions (each found exactly once)
PT-F2 CHUNKING EXPERIMENT
C1   size=300   overlap=0    chunks=475   embed=  0.5s  query=  7.0ms
C2   size=300   overlap=30   chunks=558   embed=  0.4s  query=  6.8ms
C3   size=300   overlap=60   chunks=600   embed=  0.5s  query=  6.8ms
C4   size=500   overlap=0    chunks=333   embed=  0.4s  query=  6.9ms
C5   size=500   overlap=50   chunks=339   embed=  0.4s  query=  6.9ms
C6   size=500   overlap=100  chunks=374   embed=  0.5s  query=  6.7ms
C7   size=1000  overlap=0    chunks=207   embed=  0.4s  query=  9.5ms
C8   size=1000  overlap=100  chunks=210   embed=  0.5s  query=  9.5ms
C9   size=1000  overlap=200  chunks=235   embed=  0.5s  query=  9.1ms
C10  size=1800  overlap=0    chunks=126   embed=  0.4s  query=  9.3ms
C11  size=1800  overlap=180  chunks=126   embed=  0.4s  query=  9.0ms
C12  size=1800  overlap=360  chunks=127   embed=  0.4s  query= 10.7ms

Detailed retrieval log rows: 324 (expected 12 x 9 x 3 = 324)


### Cell 2 — score the results and write the workbook

Computes Precision@3, Recall@3 and the reciprocal rank for every configuration-question pair, then aggregates them into the three tables the brief requires and writes a four-sheet workbook:

- **Table A** — configuration-level results, one row per configuration, with chunk counts and timings alongside the three metrics.
- **Table B** — the same metrics split by difficulty level, three rows per configuration.
- **Table C** — the detailed retrieval log: one row per retrieved chunk, 324 rows in all, with the source reference, relevance verdict and evidence covered, as an audit trail for the metric calculations.
- **Per-question scores** — the intermediate per-question figures that Tables A and B average over.

The detailed retrieval log is also written as a standalone CSV. Both files download automatically in Colab.

In [21]:
# ============================================================
# PT-F2: METRIC CALCULATION AND RESULT TABLES
# Precision@3, Recall@3 (over evidence units), MRR@3
# ============================================================

# ------------------------------------------------------------
# Per-question scoring
#
# Precision@3 = relevant chunks among the top 3 / 3
# Recall@3    = distinct evidence units covered by the top 3 / units required
#               (the denominator is fixed per question, so it does not drift
#               with chunk size -- this is what makes configurations comparable)
# RR          = 1 / rank of the first relevant chunk, 0 if none in the top 3
# ------------------------------------------------------------

per_question = []

for (cfg, qid), grp in detail_df.groupby(["Configuration", "Question ID"], sort=False):

    grp = grp.sort_values("Retrieved rank")

    n_required = int(grp["Evidence units required"].iloc[0])
    n_reachable = int(grp["Evidence units reachable"].iloc[0])
    difficulty = grp["Difficulty"].iloc[0]

    relevant_flags = (grp["Relevant?"] == "Yes").tolist()
    precision = sum(relevant_flags) / len(relevant_flags)

    covered = set()
    for cell in grp["Evidence covered"]:
        if isinstance(cell, str) and cell.strip():
            covered.update(part.strip() for part in cell.split(";") if part.strip())
    recall = len(covered) / n_required

    rr = 0.0
    for rank, is_rel in enumerate(relevant_flags, start=1):
        if is_rel:
            rr = 1.0 / rank
            break

    per_question.append({
        "Configuration": cfg,
        "Question ID": qid,
        "Difficulty": difficulty,
        "Units required": n_required,
        "Units reachable": n_reachable,
        "Units covered": len(covered),
        "Max Recall@3": round(min(3, n_reachable) / n_required, 4),
        "Precision@3": round(precision, 4),
        "Recall@3": round(recall, 4),
        "Reciprocal Rank": round(rr, 4),
    })

pq_df = pd.DataFrame(per_question)

# ------------------------------------------------------------
# Table A -- configuration-level results
# Overall MRR@3 is the mean reciprocal rank across ALL questions,
# never the average of the three difficulty-level means (Step 5D).
# ------------------------------------------------------------

agg = pq_df.groupby("Configuration", sort=False).agg(
    **{"Precision@3": ("Precision@3", "mean"),
       "Recall@3": ("Recall@3", "mean"),
       "MRR@3": ("Reciprocal Rank", "mean"),
       "Max Recall@3": ("Max Recall@3", "mean")}
).round(4).reset_index()

table_a = config_df.merge(agg, on="Configuration", how="left")

# ------------------------------------------------------------
# Table B -- performance by question difficulty
# ------------------------------------------------------------

table_b = (
    pq_df.groupby(["Configuration", "Difficulty"], sort=False)
         .agg(**{"Precision@3": ("Precision@3", "mean"),
                 "Recall@3": ("Recall@3", "mean"),
                 "MRR@3": ("Reciprocal Rank", "mean")})
         .round(4)
         .reset_index()
)
diff_order = {"Easy": 0, "Average": 1, "Difficult": 2}
cfg_order = {c[0]: i for i, c in enumerate(CONFIGS)}
table_b = table_b.sort_values(
    by=["Configuration", "Difficulty"],
    key=lambda s: s.map(cfg_order) if s.name == "Configuration" else s.map(diff_order)
).reset_index(drop=True)

# ------------------------------------------------------------
# Table C -- detailed retrieval log (already assembled above)
# ------------------------------------------------------------

table_c = detail_df

# ------------------------------------------------------------
# Write the workbook
# ------------------------------------------------------------

OUT_XLSX = "PT-F2_Member3_experiment_results.xlsx"
OUT_CSV = "PT-F2_Member3_detailed_retrieval_log.csv"

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    table_a.to_excel(writer, sheet_name="Table A - Configurations", index=False)
    table_b.to_excel(writer, sheet_name="Table B - By Difficulty", index=False)
    table_c.to_excel(writer, sheet_name="Table C - Retrieval Log", index=False)
    pq_df.to_excel(writer, sheet_name="Per-question scores", index=False)

    for name, frame in [("Table A - Configurations", table_a),
                        ("Table B - By Difficulty", table_b),
                        ("Table C - Retrieval Log", table_c),
                        ("Per-question scores", pq_df)]:
        sheet = writer.sheets[name]
        sheet.freeze_panes = "A2"
        for i, col in enumerate(frame.columns, start=1):
            width = max(len(str(col)) + 2,
                        min(55, int(frame[col].astype(str).str.len().max()) + 2))
            sheet.column_dimensions[sheet.cell(1, i).column_letter].width = width

print("=" * 70)
print("TABLE A - CONFIGURATION-LEVEL RESULTS")
print("=" * 70)
print(table_a[["Configuration", "Chunk size", "Overlap", "Overlap %",
               "Total chunks", "Precision@3", "Recall@3", "MRR@3",
               "Max Recall@3"]].to_string(index=False))

capped = table_a[table_a["Max Recall@3"] < 0.999]
if len(capped):
    print("\nConfigurations where some evidence is UNREACHABLE at any rank")
    print("(a unit straddles a chunk boundary and no chunk holds it whole):")
    for _, r in capped.iterrows():
        print(f"  {r['Configuration']:<4} size={r['Chunk size']:<5} overlap={r['Overlap']:<4} "
              f"max possible Recall@3 = {r['Max Recall@3']:.4f}")

table_c.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

print("\nWorkbook written:", OUT_XLSX)
print("Retrieval log written:", OUT_CSV)
print("Rows -- Table A:", len(table_a), "| Table B:", len(table_b), "| Table C:", len(table_c))

# Download in Colab
try:
    from google.colab import files
    files.download(OUT_XLSX)
    files.download(OUT_CSV)
except Exception as exc:
    print("Download skipped:", exc)

TABLE A - CONFIGURATION-LEVEL RESULTS
Configuration  Chunk size  Overlap  Overlap %  Total chunks  Precision@3  Recall@3  MRR@3  Max Recall@3
           C1         300        0          0           475       0.2222    0.2963 0.1852        0.9259
           C2         300       30         10           558       0.1111    0.1852 0.1481        0.7963
           C3         300       60         20           600       0.1852    0.2593 0.1481        0.9259
           C4         500        0          0           333       0.1111    0.2593 0.1481        0.8333
           C5         500       50         10           339       0.2222    0.3889 0.4259        0.8704
           C6         500      100         20           374       0.3704    0.5185 0.5556        0.9444
           C7        1000        0          0           207       0.4074    0.5741 0.6111        1.0000
           C8        1000      100         10           210       0.3704    0.5185 0.5000        1.0000
           C9        1000 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>